# sub045_3_bn_recalc — SWA de exp045 con BN recalculada (ver SPEC.md)

In [ ]:
SUB = "sub045_3_bn_recalc"
N_BN = 800                 # estudios de TRAIN del fold para recalcular BN
BN_BS = 4                  # estudios por batch (4 x 24 ventanas = 96 imágenes por batch BN)
BASE_PSEUDO = 0.8894654941741685   # swa_f0 de results/exp045_raptor_aug_espacial/metrics.json
BASE_GOLD = 0.9201083029612395
DELTA = 0.002
DO_REST = True             # si worked: folds 1-4 + all-data
CORPUS_OVERRIDE = "/workspace/data/cache/raptor96_384"
N_WORKERS_OVERRIDE = 8


## Setup: celdas de exp045

In [ ]:
# Ejecuta las celdas 2/4/6/8/10 del notebook de exp045 (mismo código de datos, modelo, predict y score)
import json as _json, time as _time
from pathlib import Path as _P
_here = _P.cwd(); _repo = next(p for p in [_here, *_here.parents] if (p / "src" / "knee").exists())
_src = _json.load(open(_repo / "experiments/exp045_raptor_aug_espacial/exp045_raptor_aug_espacial.ipynb", encoding="utf-8"))
_cells = ["".join(c["source"]) for c in _src["cells"]]
exec(_cells[2])
SMOKE = False; FOLDS = [0]; CORPUS = CORPUS_OVERRIDE; N_WORKERS = N_WORKERS_OVERRIDE
for _i in (4, 6, 8, 10):
    exec(_cells[_i])
import torch.nn as nn
BASE = "/workspace/users/rodrigo"
SWA_PATH = {0: f"{BASE}/wt_exp045_raptor_aug_espacial/results/exp045_raptor_aug_espacial/swa_f0.pt",
            **{k: f"{BASE}/wt_exp045_f{k}/results/exp045_raptor_aug_espacial/swa_f{k}.pt" for k in (1, 2, 3, 4)},
            "alldata": f"{BASE}/wt_exp045_raptor_aug_espacial/results/exp045_raptor_aug_espacial/swa_alldata.pt"}
OUT = _repo / "results" / SUB; OUT.mkdir(parents=True, exist_ok=True)
NONGOLD = [u for u in soft.index if u not in set(gold_ids) and u in ID2ROW]

def load_model(state):
    bb = build_backbone(ARCH, pretrained=False)
    m = RaptorClassifier(bb, F_dim=bb.num_features).to(DEVICE)
    m.load_state_dict(state); return m

def bn_sample(ids, n=None):
    ids = sorted(ids); rs = np.random.RandomState(0)
    return [ids[i] for i in sorted(rs.choice(len(ids), min(n or N_BN, len(ids)), replace=False))]

@torch.no_grad()
def recalc_bn(model, ids):
    bns = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]
    for m in bns:
        m.reset_running_stats(); m.momentum = None
    model.eval()
    for m in bns:
        m.train()
    t = _time.time()
    for x, g, idx, _sag in loader(ids, K_EVAL, False, BN_BS):
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE.type == "cuda"):
            model.encode(to_input(x, g))
    model.eval()
    print(f"BN recalculada en {len(bns)} capas con {len(ids)} estudios ({(_time.time() - t) / 60:.1f} min)", flush=True)
    return model

def evaluate(model, va_ids, name):
    t = _time.time()
    pv, pg = predict(model, va_ids), predict(model, gold_ids)
    sv, sg = score(pv), score(pg)
    if len(pv): pv.add_prefix("pred_").reset_index().to_csv(OUT / f"oof_{name}.csv", index=False)
    pg.add_prefix("pred_").reset_index().to_csv(OUT / f"gold_{name}.csv", index=False)
    r = {"pseudo_auc": sv["pseudo"]["mean"] if sv["pseudo"] else None, "v1_auc": sv["v1"]["mean"] if sv["v1"] else None,
         "gold_auc": sg["gold"]["mean"], "pseudo_per_label": sv["pseudo"]["per_label"] if sv["pseudo"] else None,
         "gold_per_label": sg["gold"]["per_label"], "n_val": len(va_ids), "minutes": (_time.time() - t) / 60}
    print(f"[{name}] pseudo L2 {r['pseudo_auc'] or float('nan'):.4f} | v1 {r['v1_auc'] or float('nan'):.4f} | gold {r['gold_auc']:.4f} | {r['minutes']:.1f} min", flush=True)
    return r

def save_metrics(m):
    m.update({"schema_version": 1, "exp": SUB, "padre": "exp045_raptor_aug_espacial", "author": AUTHOR, "git": GIT,
              "written_utc": datetime.now(timezone.utc).isoformat()})
    (OUT / "metrics.json").write_text(_json.dumps(m, indent=1, default=float))
    print("metrics ->", OUT / "metrics.json", flush=True)


## fold 0 (decide)

In [ ]:
M = {"criterio": {"worked_if_pseudo_ge": BASE_PSEUDO + DELTA, "gold_veto_lt": BASE_GOLD - 0.02, "base_pseudo": BASE_PSEUDO, "base_gold": BASE_GOLD},
     "n_bn": N_BN, "per_fold": {}}
ck = torch.load(SWA_PATH[0], map_location="cpu", weights_only=False)
tr0, va0 = fold_ids(0)
model = load_model(ck["model"])
M["sanity_gold_swa_f0_sin_cambio"] = score(predict(model, gold_ids))["gold"]["mean"]
print("sanidad gold swa_f0 (esperado 0.92011):", M["sanity_gold_swa_f0_sin_cambio"], flush=True)
recalc_bn(model, bn_sample(tr0))
r = evaluate(model, va0, "swa_bn_f0")
M["per_fold"]["0"] = r
M["delta_pseudo_f0"] = r["pseudo_auc"] - BASE_PSEUDO
M["delta_gold_f0"] = r["gold_auc"] - BASE_GOLD
VERDICT = "killed" if r["gold_auc"] < BASE_GOLD - 0.02 else ("worked" if r["pseudo_auc"] >= BASE_PSEUDO + DELTA else "didnt")
M["verdict"] = VERDICT
torch.save({**{k: v for k, v in ck.items() if k != "model"}, "model": model.state_dict(), "bn_recalc": {"n": N_BN, "sub": SUB}},
           SWA_PATH[0].replace("swa_f0.pt", "swa_bn_f0.pt"))
save_metrics(M)
print(f"VERDICT {VERDICT} · Δ pseudo {M['delta_pseudo_f0']:+.4f} · Δ gold {M['delta_gold_f0']:+.4f}", flush=True)
del model; torch.cuda.empty_cache()


## folds 1-4 + all-data (solo si worked)

In [ ]:
if VERDICT == "worked" and DO_REST:
    for k in (1, 2, 3, 4, "alldata"):
        ck = torch.load(SWA_PATH[k], map_location="cpu", weights_only=False)
        model = load_model(ck["model"])
        if k == "alldata":
            recalc_bn(model, bn_sample(NONGOLD)); va = []
        else:
            tr, va = fold_ids(k); recalc_bn(model, bn_sample(tr))
        dst = SWA_PATH[k].replace("swa_", "swa_bn_")
        torch.save({**{kk: v for kk, v in ck.items() if kk != "model"}, "model": model.state_dict(), "bn_recalc": {"n": N_BN, "sub": SUB}}, dst)
        M["per_fold"][str(k)] = {**evaluate(model, va, f"swa_bn_{'alldata' if k == 'alldata' else 'f%s' % k}"), "ckpt": dst}
        save_metrics(M); del model; torch.cuda.empty_cache()
print(_json.dumps({k: (v.get("pseudo_auc"), v["gold_auc"]) for k, v in M["per_fold"].items()}))
